Here we look into the metadata and try to analyze induviduals and different cells the samples are from

In [ ]:
import pandas as pd

In [ ]:
samples_df = pd.read_csv(r'..\data\sample_metadata.tsv', sep='\t')
individuals_df = pd.read_csv(r'..\data\individual_metadata.tsv', sep='\t')
samples_to_individuals_df = pd.read_csv(r'..\data\sample_individual_mapping.tsv', sep='\t')


In [ ]:
samples_df.head()

In [ ]:
samples_df.shape

In [ ]:
samples_df['biopsy_type'].value_counts()

In [ ]:
individuals_df.head()

In [ ]:
individuals_df.shape

In [ ]:
individuals_df['disease_status'].value_counts()

In [ ]:
individuals_df['age_binned_10'].value_counts().sort_index().plot(kind='bar', title='Number of individuals by age group')

In [ ]:
individuals_df['sex'].value_counts()

In [ ]:
individuals_df.groupby('disease_status')['ti_ses_inflamed'].value_counts()

In [ ]:
individuals_df[individuals_df['disease_status'] == 'CD'].groupby('ti_ses_inflamed')['sex'].value_counts()

In [ ]:
individuals_df.groupby('disease_status')['sex'].value_counts()

In [ ]:
individuals_df.groupby('disease_status')['age_binned_10'].agg(['count', 'mean', 'min', 'max', 'median'])

In [ ]:
samples_to_individuals_df.head()

In [ ]:
samples_to_individuals_df.shape

Check if the samples-to individual file contains all samples, and individual from samples_df and individuals_df

In [ ]:
sample_match_count = (samples_df['sanger_sample_id'].isin(samples_to_individuals_df['sanger_sample_id'])).sum()
Missing_sample_count = (~samples_df['sanger_sample_id'].isin(samples_to_individuals_df['sanger_sample_id'])).sum()

individual_match_count = (individuals_df['individual_id'].isin(samples_to_individuals_df['individual_id'])).sum()
missing_individual_count = (~individuals_df['individual_id'].isin(samples_to_individuals_df['individual_id'])).sum()

print(f"matching samples: {sample_match_count}")
print(f"matching individuals: {individual_match_count}")

print(f"Number of samples in sample metadata that do not have a match in the mapping file: {Missing_sample_count}")
print(f"Number of individuals in individual metadata that do not have a match in the mapping file: {missing_individual_count}")

In [ ]:
# find what samples are missing in the mapping file
missing_samples = samples_df[~samples_df['sanger_sample_id'].isin(samples_to_individuals_df['sanger_sample_id'])]
missing_samples

In [ ]:
# find what individuals are missing in the mapping file
missing_individuals = individuals_df[~individuals_df['individual_id'].isin(samples_to_individuals_df['individual_id'])]
missing_individuals

In [ ]:
samples_to_individuals_df.loc[samples_to_individuals_df['sanger_sample_id'] == missing_samples.iloc[0]['sanger_sample_id']]

In [ ]:
samples_to_individuals_df.loc[samples_to_individuals_df['individual_id'] == missing_individuals.iloc[0]['individual_id']]

In [ ]:
temp_mapping = pd.merge(samples_df, samples_to_individuals_df, on='sanger_sample_id', how='left')
mapping_df = pd.merge(temp_mapping, individuals_df, on='individual_id', how='left')

In [ ]:
mapping_df.shape

In [ ]:
mapping_df.head()

In [ ]:
mapping_df[mapping_df['individual_id'].isnull()]

In [ ]:
mapping_df.groupby('biopsy_type')['disease_status'].value_counts()

In [ ]:
individuals_df.groupby(['disease_status', 'ti_ses_inflamed'])['age_binned_10'].describe()

In [ ]:
mapping_df.groupby(['disease_status', 'ti_ses_inflamed'])['age_binned_10'].describe()

In [ ]:
mapping_df.groupby(['disease_status', 'ti_ses_inflamed'])['age_binned_10'].value_counts().sort_index().plot(kind='bar', title='Age distribution by disease status and inflammation status')

In [ ]:
def convert_to_ratio(value: str):
    if value:
        s = value.split(':')
        ratio = ':'.join(str(int(i)) for i in s[:2])
        return ratio

f2_f1 ratio is in hh:mm:ss format. We will revert it to ratio format

In [ ]:
mapping_df['f2_f1_ratio'] = mapping_df['f2_f1_ratio'].map(convert_to_ratio, na_action='ignore')
mapping_df['f2_f1_ratio'].value_counts()

In [ ]:
mapping_df.notna().sum()

In [ ]:
samples_df.isna().sum()

In [ ]:
individuals_df.notna().sum()

Combining ti_ses_inflamed and disease_status into one because healthy always has uninfamed gut as shown in metadata anlysis

In [ ]:
mapping_df['health_category'] = list(zip(mapping_df['disease_status'], mapping_df['ti_ses_inflamed']))
mapping_df['health_category'].value_counts()

In [ ]:
mapping_df['health_category'] = mapping_df['health_category'].map({
    ('Healthy', 'uninflamed'): 'Healthy',
    ('CD', 'uninflamed'): 'CD uninflamed',
    ('CD', 'inflamed'): 'CD inflamed'})
mapping_df['health_category'].value_counts()

In [ ]:
# check if missing health category is due to missing individual id
mapping_df['health_category'].isnull().sum()

In [ ]:
mapping_df.to_csv(r'..\data\mapped_data.tsv', sep='\t', index=None)

In [ ]:
donors = mapping_df[['individual_id', 'health_category', 'f2_f1_ratio']].copy()
donors['f2_f1_ratio'] = donors['f2_f1_ratio'].astype(object).fillna('Unknown')
crosstab = pd.crosstab(donors['f2_f1_ratio'], donors['health_category'])
crosstab.drop('Unknown', inplace=True)
crosstab.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='Set2', xlabel='f2_f1', ylabel='Number of Donors', title='Distribution of f2_f1 Ratio by Health Category')

4:1 ratio has no healthy donors.  
1:0 is the most prevalent.

In [ ]:
donors = mapping_df[['individual_id', 'health_category', 'bead_version']].copy()
donors['bead_version'] = donors['bead_version'].astype(object).fillna('Unknown')
crosstab = pd.crosstab(donors['bead_version'], donors['health_category'])
crosstab.plot(kind='bar', figsize=(10, 6), colormap='Set2', xlabel='Bead Version', ylabel='Number of Donors', title='Distribution of Bead Version by Health Category')

In [ ]:
donors = mapping_df[['individual_id', 'health_category', 'chip_version']].copy()
donors['chip_version'] = donors['chip_version'].astype(object).fillna('Unknown')
crosstab = pd.crosstab(donors['chip_version'], donors['health_category'])
crosstab.plot(kind='bar', figsize=(10, 6), colormap='Set2', xlabel='Chip Version', ylabel='Number of Donors', title='Distribution of Chip Version by Health Category')

chip b has very less donors, but most of them have CD inflamed. On the otehr hand, chip g has a bigger share of donors with over 200 healthy people, compared to less than 100 donors with CD(inflamed+uninflamed).  
Similar phenomenon is observed with bead version with v3 having predominantly CD inflamed and v3.1 has=ving more than double of healthy donors

### Technical variables vs. disease group: summary table
One row per technical column, counted in **donors** (not cells). For each column:
- how many levels (versions / lots) it has and how many donors sit in each level
- how many donors have no value
- how many levels contain only one health category (those levels cannot separate batch from disease). Levels with very few donors trivially have one group, so we also count them among levels with at least 5 donors.

In [ ]:
# one row per donor; drop the sample with no donor information
donor_table = mapping_df.dropna(subset=['individual_id']).copy()
donor_table['is_cd'] = donor_table['disease_status'] == 'CD'
assert donor_table['individual_id'].is_unique

technical_cols = ['bead_version', 'chip_version', 'bead_lot', 'chip_lot', 'gem_lot', 'enzyme_lot_BLP', 'f2_f1_ratio']
MIN_DONORS = 5  # levels with fewer donors are too small to say anything about

rows = []
for col in technical_cols:
    known = donor_table.dropna(subset=[col])
    donors_per_level = known.groupby(col)['individual_id'].nunique()
    groups_per_level = known.groupby(col)['health_category'].nunique()
    disease_per_level = known.groupby(col)['disease_status'].nunique()
    big_levels = donors_per_level >= MIN_DONORS
    cd_share = known.groupby(col)['is_cd'].mean()[big_levels]
    rows.append({
        'column': col,
        'levels': len(donors_per_level),
        'donors per level (min)': donors_per_level.min(),
        'donors per level (median)': donors_per_level.median(),
        'donors per level (max)': donors_per_level.max(),
        'missing donors': donor_table[col].isna().sum(),
        'single-group levels': (groups_per_level == 1).sum(),
        f'levels with >={MIN_DONORS} donors': big_levels.sum(),
        'single-group levels among those': ((groups_per_level == 1) & big_levels).sum(),
        'levels with only one disease': ((disease_per_level == 1) & big_levels).sum(),
        'CD share (min)': cd_share.min(),
        'CD share (max)': cd_share.max(),
    })

technical_summary = pd.DataFrame(rows).set_index('column')
technical_summary

Inference:  
1. chip version has imbalanced groups
2. fragemented lots: (bead, chip, gem) have a lot of smaller groups, around 32 to 40 groups with a median donor size of 4.5 to 6.5
3. f2_f1 has huge missing values. 274 out of 342 donors are missing. It cannot be used. 
4. we can see that when donors more than 5 groups are considered, mutiple health category data is present. Although, not distributed equally as indicated by CD share
5. emzyme lot is the only column that is decently balanced regarding CD share